# Laboratoire 01 : Ingestion Bronze Multi-Sources & Déversement des Données (Staging)
**Cours :** Business Intelligence et Analyse des Mégadonnées (`01 31 0367 00`)  
**Public cible :** Étudiants ingénieurs en 4e année B.Sc. (Génie Industriel et Management de la Production)  
**Durée :** 2 Heures (120 Minutes)  
**Rythme :** Séance pratique de laboratoire du mardi

---

## 🎯 1. Contexte & Objectifs
En ingénierie moderne des données industrielles, les flux bruts issus des réseaux de supervision SCADA d'atelier, des journaux de maintenance et des bases de données ERP arrivent sous des formats hétérogènes, avec des schémas et des fréquences d'échantillonnage variables.

Dans ce laboratoire, vous allez mettre en place la **Couche Bronze** de votre architecture Médaillon :
1. Localiser et inspecter les fichiers de données brutes correspondant à l'étude de cas attribuée à votre équipe :
   * Cas n° 1 : `case1_smartpress_automotive` (télémesure CSV, tickets d'intervention JSON, données de référence Excel)
   * Cas n° 2 : `case2_dataco_supply_chain` (commandes de vente CSV, événements de transport JSON, catalogue de référence Excel)
   * Cas n° 3 : `case3_nexis_workforce` (production par poste CSV, événements RH JSON, effectifs et compétences Excel)
2. Ingérer les fichiers CSV, JSON et les classeurs Excel multi-onglets dans des DataFrames `pandas`.
3. Ajouter les colonnes d'**audit et de traçabilité d'entreprise (*Audit Lineage*)** :
   * `_ingestion_timestamp_utc` (horodatage ISO 8601 en UTC)
   * `_source_file` (nom du fichier source brut d'origine)
4. Déverser (*stage*) les données brutes sans altération dans des **fichiers colonnaires Parquet** immuables au sein du dossier `data/bronze_staged/`.

> **La Règle d'Or du Bronze :** Ne modifiez, ne filtrez et ne nettoyez jamais les mesures physiques dans la couche Bronze ! Le Bronze est une réplique exacte et auditable de la réalité opérationnelle de l'atelier. Le nettoyage, l'exclusion des valeurs aberrantes et l'imputation s'effectuent dans la **couche Silver (Laboratoires 02 & 03)**.


## ⚙️ 2. Configuration de l'Environnement & Résolution Dynamique des Chemins
Importez d'abord les bibliothèques requises et déterminez dynamiquement la racine du dépôt ainsi que les répertoires de votre étude de cas.


In [ ]:
# Étape 1 : Importations
import json
from datetime import datetime, timezone
from pathlib import Path
try:
    import pandas as pd
except ImportError:
    pd = None

# Étape 2 : Détection dynamique de la racine et des données
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'requirements.txt').exists() and not (REPO_ROOT / '02_data_generator').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Étape 3 : Configuration des chemins de l'étude de cas (Par défaut : Cas n° 1 SmartPress Automotive)
# (Si votre équipe travaille sur le Cas 2 ou 3, adaptez le nom du dossier ci-dessous !)
CASE_NAME = 'case1_smartpress_automotive'

# Vérifier si les données sont directement dans lab_01/data ou 05_case_studies
LAB_DIR = Path.cwd().resolve()
if not (LAB_DIR / 'data').exists() and (REPO_ROOT / '03_labs_student_starters' / 'lab_01_bronze_ingestion' / 'data').exists():
    LAB_DIR = REPO_ROOT / '03_labs_student_starters' / 'lab_01_bronze_ingestion'

if (LAB_DIR / 'data' / CASE_NAME).exists():
    CASE_DIR = LAB_DIR / 'data' / CASE_NAME
    BRONZE_RAW = CASE_DIR / 'bronze' if (CASE_DIR / 'bronze').exists() else CASE_DIR
    BRONZE_STAGED = CASE_DIR / 'bronze_staged'
else:
    CASE_DIR = REPO_ROOT / '05_case_studies_reference_solutions' / CASE_NAME
    BRONZE_RAW = CASE_DIR / 'data' / 'bronze'
    BRONZE_STAGED = CASE_DIR / 'data' / 'bronze_staged'

# S'assurer que le répertoire de transit (staging) existe
BRONZE_STAGED.mkdir(parents=True, exist_ok=True)

print(f"Racine du dépôt : {REPO_ROOT}")
print(f"Étude de cas sélectionnée : {CASE_NAME}")
print(f"Le dossier Bronze brut existe : {BRONZE_RAW.exists()}")
print(f"Dossier de transit (Staged) : {BRONZE_STAGED}")


## 📥 3. Partie 1 : Ingestion du flux CSV de télémesure & Ajout de la traçabilité d'audit
Chargez à présent le fichier CSV brut (`sensor_telemetry.csv` pour SmartPress, `order_items_trans.csv` pour DataCo, ou `shift_production_performance.csv` pour Nexis).

### Votre Mission :
1. Lire le fichier CSV dans un DataFrame nommé `df_telemetry`.
2. Générer l'horodatage UTC actuel sous forme de chaîne ISO-8601 : `datetime.now(timezone.utc).isoformat()`.
3. Ajouter les colonnes `_ingestion_timestamp_utc` et `_source_file` au DataFrame.
4. Afficher les dimensions `.shape`, les types de colonnes via `.info()` et prévisualiser les 5 premières lignes avec `.head()`.


In [ ]:
# À FAIRE : Implémentation Étudiant - Ingestion CSV
now_utc = datetime.now(timezone.utc).isoformat()
csv_file_name = 'sensor_telemetry.csv'  # À adapter si Cas 2 ou 3
csv_path = BRONZE_RAW / csv_file_name

# 1. Lire le CSV via pandas
# df_telemetry = ...

# 2. Rattacher les colonnes d'audit
# df_telemetry['_ingestion_timestamp_utc'] = ...
# df_telemetry['_source_file'] = ...

# 3. Inspecter les dimensions et prévisualiser les enregistrements
# print(f"Ingestion réussie de {len(df_telemetry)} enregistrements.")
# df_telemetry.head()


## 📋 4. Partie 2 : Ingestion des journaux de maintenance semi-structurés JSON
Chargez la source JSON (`maintenance_logs.json` pour SmartPress, `shipment_events.json` pour DataCo, ou `workforce_events.json` pour Nexis).

### Votre Mission :
1. Ouvrir et charger le fichier JSON à l'aide de la fonction standard `json.load()`.
2. Convertir la liste d'enregistrements JSON en un DataFrame tabulaire nommé `df_maintenance`.
3. Rattacher les colonnes de traçabilité d'audit (`_ingestion_timestamp_utc` et `_source_file`).
4. Afficher les dimensions et prévisualiser les 5 premiers tickets de maintenance.


In [ ]:
# À FAIRE : Implémentation Étudiant - Ingestion JSON
json_file_name = 'maintenance_logs.json'
json_path = BRONZE_RAW / json_file_name

# 1. Charger le fichier JSON via json.load()
# with open(json_path, 'r', encoding='utf-8') as f:
#     json_data = ...

# 2. Convertir en DataFrame pandas
# df_maintenance = ...

# 3. Rattacher les colonnes d'audit
# df_maintenance['_ingestion_timestamp_utc'] = ...
# df_maintenance['_source_file'] = ...

# 4. Prévisualiser
# print(f"Ingestion réussie de {len(df_maintenance)} tickets de maintenance.")
# df_maintenance.head()


## 📊 5. Partie 3 : Ingestion des données de référence issues d'un classeur Excel multi-onglets
Les systèmes opérationnels gèrent couramment les données de référence de l'entreprise (catalogue de machines, plannings d'équipes, nomenclatures de pièces, segments clients) dans des classeurs Excel à feuilles multiples (`machines_and_shifts.xlsx` ou `master_catalog.xlsx`).

### Votre Mission :
1. Inspecter les feuilles disponibles à l'aide de `pd.ExcelFile`.
2. Charger chaque feuille dans son propre DataFrame dédié (ex. `df_machines` et `df_shifts`).
3. Rattacher les colonnes d'audit aux deux DataFrames.
4. Prévisualiser les schémas et le contenu des deux tables.


In [ ]:
# À FAIRE : Implémentation Étudiant - Ingestion Excel Multi-Feuilles
excel_file_name = 'machines_and_shifts.xlsx'
excel_path = BRONZE_RAW / excel_file_name

# 1. Inspecter le nom des feuilles via pd.ExcelFile
# excel_file = pd.ExcelFile(excel_path)
# print("Feuilles disponibles :", excel_file.sheet_names)

# 2. Charger chaque feuille dans un DataFrame dédié
# df_machines = pd.read_excel(excel_path, sheet_name='Machines')
# df_shifts = pd.read_excel(excel_path, sheet_name='Shifts')

# 3. Rattacher les colonnes d'audit aux deux DataFrames
# df_machines['_ingestion_timestamp_utc'] = now_utc
# df_machines['_source_file'] = excel_file_name
# df_shifts['_ingestion_timestamp_utc'] = now_utc
# df_shifts['_source_file'] = excel_file_name

# 4. Prévisualiser les DataFrames
# display(df_machines)
# display(df_shifts)


## 💾 6. Partie 4 : Déversement au format colonnaire Parquet (Staging)
Maintenant que l'ensemble des sources hétérogènes est chargé en mémoire vive et estampillé avec les métadonnées d'audit, déversez-les sous forme de **fichiers Parquet immuables** dans `data/bronze_staged/`.

### Pourquoi le format Parquet ?
* Contrairement au CSV, Parquet est un **format de stockage colonnaire** qui intègre le schéma et les types de données (horodatages, flottants, entiers) directement dans les métadonnées de fichier.
* Il applique une compression performante (Snappy), réduisant considérablement l'espace de stockage disque.
* Les moteurs analytiques en aval comme **DuckDB** peuvent interroger directement les fichiers Parquet sans copie en mémoire vive (*zero-copy scans*).

### Votre Mission :
1. Exporter `df_telemetry`, `df_maintenance`, `df_machines` et `df_shifts` au format Parquet avec `.to_parquet(chemin, index=False)`.
2. Vérifier que les fichiers sont bien créés sur le disque et afficher leur taille.


In [ ]:
# À FAIRE : Implémentation Étudiant - Export Parquet
# Veillez à spécifier index=False pour éviter de persister les index entiers arbitraires de pandas !

out_telem = BRONZE_STAGED / 'bronze_telemetry.parquet'
out_maint = BRONZE_STAGED / 'bronze_maintenance.parquet'
out_mach = BRONZE_STAGED / 'bronze_machines.parquet'
out_shifts = BRONZE_STAGED / 'bronze_shifts.parquet'

# df_telemetry.to_parquet(out_telem, index=False)
# df_maintenance.to_parquet(out_maint, index=False)
# df_machines.to_parquet(out_mach, index=False)
# df_shifts.to_parquet(out_shifts, index=False)

# Vérification sur disque :
# for p in [out_telem, out_maint, out_mach, out_shifts]:
#     print(f"Fichier créé : {p.name} ({p.stat().st_size / 1024:.2f} Ko)")


## 🔍 7. Inspection Exploratoire : Détection des Anomalies Injectées
Avant de conclure ce Laboratoire 01, prenez 5 minutes pour explorer les données déversées et observer les anomalies qui feront l'objet de vos traitements de nettoyage lors des Laboratoires 02 et 03 !

Exécutez la cellule exploratoire ci-dessous pour repérer ces anomalies.


In [ ]:
# Inspection exploratoire des anomalies (Aperçu pour les Laboratoires 02 & 03)
# df_test = pd.read_parquet(out_telem)
# print("Identifiants uniques de machines dans la télémesure brute :")
# print(df_test['machine_id'].unique())
# print("\nStatistiques descriptives du couple mécanique (observez le min et le max !) :")
# print(df_test['torque_Nm'].describe())
# print("\nNombre de valeurs manquantes :")
# print(df_test.isna().sum())
